# Find Best Model from Grid Search Results

This notebook analyzes the grid search results JSON file to find the model parameters that achieved the best final IoU score.

In [2]:
# Import Required Libraries
import json
import pandas as pd
import numpy as np
import os

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
# Load Grid Results JSON File
grid_results_path = "guided_box_ijmond/grid_search/grid_results.json"

# Check if file exists
if os.path.exists(grid_results_path):
    with open(grid_results_path, 'r') as f:
        grid_results = json.load(f)
    print(f"✓ Loaded grid results from: {grid_results_path}")
    print(f"  Number of experiments: {len(grid_results)}")
else:
    print(f"❌ File not found: {grid_results_path}")
    print("Available files in current directory:")
    for item in os.listdir("."):
        print(f"  {item}")

✓ Loaded grid results from: guided_box_ijmond/grid_search/grid_results.json
  Number of experiments: 180


In [4]:
# Parse and Extract Model Parameters
# Convert results to DataFrame for easy analysis
if 'grid_results' in locals():
    # Extract key information from each experiment
    data = []
    for result in grid_results:
        experiment = {
            'run_name': result.get('run_name', 'unknown'),
            'learning_rate': result.get('learning_rate', None),
            'ema_alpha': result.get('ema_alpha', None),
            'img_size': result.get('img_size', None),
            'positive_weight': result.get('positive_weight', None),
            'best_iou': result.get('best_iou', None),
            'final_iou': result.get('final_iou', None),
            'final_val_loss': result.get('final_val_loss', None),
            'time_seconds': result.get('time_seconds', None)
        }
        data.append(experiment)
    
    # Create DataFrame
    df = pd.DataFrame(data)
    print("Grid search results summary:")
    print(f"  Total experiments: {len(df)}")
    print(f"  Parameter ranges:")
    print(f"    Learning rate: {df['learning_rate'].min():.6f} - {df['learning_rate'].max():.6f}")
    print(f"    EMA alpha: {df['ema_alpha'].min():.3f} - {df['ema_alpha'].max():.3f}")
    print(f"    Image size: {df['img_size'].min()} - {df['img_size'].max()}")
    print(f"    Positive weight: {df['positive_weight'].min():.1f} - {df['positive_weight'].max():.1f}")
    print(f"  IoU range:")
    print(f"    Best IoU: {df['best_iou'].min():.4f} - {df['best_iou'].max():.4f}")
    print(f"    Final IoU: {df['final_iou'].min():.4f} - {df['final_iou'].max():.4f}")
    
    # Display first few rows
    print("\nFirst 5 experiments:")
    print(df.head().to_string(index=False))

Grid search results summary:
  Total experiments: 180
  Parameter ranges:
    Learning rate: 0.000010 - 0.000500
    EMA alpha: 0.850 - 0.950
    Image size: 128.0 - 512.0
    Positive weight: 1.5 - 9.0
  IoU range:
    Best IoU: 0.0000 - 0.1616
    Final IoU: 0.0000 - 0.1543

First 5 experiments:
                   run_name  learning_rate  ema_alpha  img_size  positive_weight  best_iou  final_iou  final_val_loss  time_seconds
lr0.0001_ema0.85_img128_pw3         0.0001       0.85     128.0              3.0  0.087248   0.075276        0.273551     29.664879
lr0.0001_ema0.85_img128_pw5         0.0001       0.85     128.0              5.0  0.089165   0.073143        0.360153     26.874976
lr0.0001_ema0.85_img128_pw7         0.0001       0.85     128.0              7.0  0.085353   0.082300        0.430428     26.842671
lr0.0001_ema0.85_img128_pw9         0.0001       0.85     128.0              9.0  0.086384   0.077044        0.449358     27.192030
lr0.0001_ema0.85_img256_pw3         0.000

In [9]:
best_iou_idx = df['best_iou'].idxmax()
best_iou_idx

20

In [16]:
max_best_iou = df['best_iou'].max()
max_best_iou

np.float64(0.16160474310163409)

In [17]:
max_final_iou = df['final_iou'].max()
max_final_iou

np.float64(0.15428839686454363)

In [12]:
df.head()

,run_name,learning_rate,ema_alpha,img_size,positive_weight,best_iou,final_iou,final_val_loss,time_seconds
0,lr0.0001_ema0.85_img128_pw3,0.0001,0.85,128.0,3.0,0.087248,0.075276,0.273551,29.664879
1,lr0.0001_ema0.85_img128_pw5,0.0001,0.85,128.0,5.0,0.089165,0.073143,0.360153,26.874976
2,lr0.0001_ema0.85_img128_pw7,0.0001,0.85,128.0,7.0,0.085353,0.082300,0.430428,26.842671
3,lr0.0001_ema0.85_img128_pw9,0.0001,0.85,128.0,9.0,0.086384,0.077044,0.449358,27.192030
4,lr0.0001_ema0.85_img256_pw3,0.0001,0.85,256.0,3.0,0.117396,0.088021,0.265845,30.811360


In [14]:
df.loc[20]

run_name           lr0.0001_ema0.9_img512_pw3
learning_rate                          0.0001
ema_alpha                                 0.9
img_size                                512.0
positive_weight                           3.0
best_iou                             0.161605
final_iou                            0.089801
final_val_loss                       0.265442
time_seconds                        60.925806
Name: 20, dtype: object

In [5]:
# Find Best IoU Score
if 'df' in locals():
    # Find best final IoU
    best_final_iou_idx = df['final_iou'].idxmax()
    best_final_iou = df.loc[best_final_iou_idx, 'final_iou']
    
    # Find best IoU (could be different from final)
    best_iou_idx = df['best_iou'].idxmax()
    best_iou = df.loc[best_iou_idx, 'best_iou']
    
    print("🔍 BEST IoU ANALYSIS")
    print("=" * 50)
    print(f"Best Final IoU: {best_final_iou:.4f} (Experiment #{best_final_iou_idx + 1})")
    print(f"Best IoU Overall: {best_iou:.4f} (Experiment #{best_iou_idx + 1})")
    print("")
    
    if best_final_iou_idx != best_iou_idx:
        print("⚠️  Note: Best final IoU is different from best IoU overall")
        print("   This suggests the best model may have overfitted during training")
    else:
        print("✓ Best final IoU matches best IoU overall - good stability!")

🔍 BEST IoU ANALYSIS
Best Final IoU: 0.1543 (Experiment #115)
Best IoU Overall: 0.1616 (Experiment #21)

⚠️  Note: Best final IoU is different from best IoU overall
   This suggests the best model may have overfitted during training


In [11]:
# Extract Best Model Parameters (Final IoU)
if 'df' in locals() and 'best_final_iou_idx' in locals():
    best_model = df.loc[best_final_iou_idx]
    
    print("🏆 BEST MODEL PARAMETERS (Best Final IoU)")
    print("=" * 60)
    print(f"Run Name: {best_model['run_name']}")
    print("")
    print("Hyperparameters:")
    print(f"  • Learning Rate:    {best_model['learning_rate']:.6f}")
    print(f"  • EMA Alpha:        {best_model['ema_alpha']:.3f}")
    print(f"  • Image Size:       {best_model['img_size']}")
    print(f"  • Positive Weight:  {best_model['positive_weight']:.1f}")
    print("")
    print("Performance:")
    print(f"  • Final IoU:        {best_model['final_iou']:.4f}")
    print(f"  • Best IoU:         {best_model['best_iou']:.4f}")
    print(f"  • Final Val Loss:   {best_model['final_val_loss']:.4f}")
    print(f"  • Training Time:    {best_model['time_seconds']:.1f} seconds")
    
    # Calculate IoU stability (difference between best and final)
    iou_stability = best_model['final_iou'] - best_model['best_iou']
    print("")
    print("Stability Analysis:")
    print(f"  • IoU Drop:         {abs(iou_stability):.4f}")
    if iou_stability >= -0.01:
        print("  • Stability:        ✓ Good (minimal overfitting)")
    elif iou_stability >= -0.05:
        print("  • Stability:        ⚠️  Moderate (some overfitting)")
    else:
        print("  • Stability:        ❌ Poor (significant overfitting)")

🏆 BEST MODEL PARAMETERS (Best Final IoU)
Run Name: resnet50_3layer_ep10_lr1e-05_pw1.5

Hyperparameters:
  • Learning Rate:    0.000010
  • EMA Alpha:        nan
  • Image Size:       nan
  • Positive Weight:  1.5

Performance:
  • Final IoU:        0.1543
  • Best IoU:         0.1543
  • Final Val Loss:   0.1370
  • Training Time:    57.3 seconds

Stability Analysis:
  • IoU Drop:         0.0000
  • Stability:        ✓ Good (minimal overfitting)


In [7]:
# Display Results Summary
if 'df' in locals():
    # Sort by final IoU for ranking
    df_sorted = df.sort_values('final_iou', ascending=False).reset_index(drop=True)
    
    print("📊 TOP 10 MODELS (by Final IoU)")
    print("=" * 80)
    
    # Display top 10 models
    top_models = df_sorted.head(10)[['run_name', 'learning_rate', 'ema_alpha', 'img_size', 
                                    'positive_weight', 'final_iou', 'best_iou', 'final_val_loss']]
    
    # Format for better display
    top_models_display = top_models.copy()
    top_models_display['rank'] = range(1, len(top_models_display) + 1)
    top_models_display = top_models_display[['rank', 'run_name', 'learning_rate', 'ema_alpha', 
                                           'img_size', 'positive_weight', 'final_iou', 'best_iou', 'final_val_loss']]
    
    # Round numerical columns for better readability
    top_models_display['learning_rate'] = top_models_display['learning_rate'].round(6)
    top_models_display['ema_alpha'] = top_models_display['ema_alpha'].round(3)
    top_models_display['final_iou'] = top_models_display['final_iou'].round(4)
    top_models_display['best_iou'] = top_models_display['best_iou'].round(4)
    top_models_display['final_val_loss'] = top_models_display['final_val_loss'].round(4)
    
    print(top_models_display.to_string(index=False))
    
    print("\n" + "=" * 80)
    print("💡 ANALYSIS SUMMARY:")
    print(f"   • {len(df)} total experiments completed")
    print(f"   • Best final IoU: {df['final_iou'].max():.4f}")
    print(f"   • Worst final IoU: {df['final_iou'].min():.4f}")
    print(f"   • Average final IoU: {df['final_iou'].mean():.4f}")
    print(f"   • IoU standard deviation: {df['final_iou'].std():.4f}")
    
    # Find best parameter combinations
    print(f"\n🎯 OPTIMAL PARAMETER RANGES (Top 5 models):")
    top_5 = df_sorted.head(5)
    print(f"   • Learning Rate: {top_5['learning_rate'].min():.6f} - {top_5['learning_rate'].max():.6f}")
    print(f"   • EMA Alpha: {top_5['ema_alpha'].min():.3f} - {top_5['ema_alpha'].max():.3f}")
    print(f"   • Image Size: {top_5['img_size'].min()} - {top_5['img_size'].max()}")
    print(f"   • Positive Weight: {top_5['positive_weight'].min():.1f} - {top_5['positive_weight'].max():.1f}")

📊 TOP 10 MODELS (by Final IoU)
 rank                            run_name  learning_rate  ema_alpha  img_size  positive_weight  final_iou  best_iou  final_val_loss
    1  resnet50_3layer_ep10_lr1e-05_pw1.5        0.00001        NaN       NaN              1.5     0.1543    0.1543          0.1370
    2          lr0.0005_ema0.9_img512_pw5        0.00050       0.90     512.0              5.0     0.1345    0.1345          0.2303
    3         lr0.0001_ema0.95_img512_pw9        0.00010       0.95     512.0              9.0     0.1149    0.1497          0.4536
    4         lr0.0002_ema0.85_img512_pw3        0.00020       0.85     512.0              3.0     0.1099    0.1573          0.2443
    5         lr0.0001_ema0.85_img512_pw7        0.00010       0.85     512.0              7.0     0.1097    0.1406          0.3329
    6  resnet50_4layer_ep10_lr1e-05_pw2.5        0.00001        NaN       NaN              2.5     0.1085    0.1250          0.2314
    7  resnet50_3layer_ep10_lr1e-05_pw2.5    